# Rule: **build_energy_totals**

**Outputs**

- resources/`transformation_output_coke.csv`
- resources/`energy_totals.csv`
- resources/`co2_totals.csv`
- resources/`transport_data.csv`
- resources/`district_heat_share.csv`
- resources/`heating_efficiencies.csv`

In [ ]:
######################################## Parameters

### Run
name = ''
prefix = ''


In [ ]:
##### Import packages
import matplotlib.pyplot as plt
import os 
import sys
import numpy as np


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `transformation_output_coke.csv`

Load the file and preview its content.

In [ ]:
file = "transformation_output_coke.csv"

df_coke = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df_coke.head()

## `energy_totals.csv`

Load the file and preview its content.

In [ ]:
file = "energy_totals.csv"

df_energy_totals = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df_energy_totals.head()

### `residential`

In [ ]:
#################### Parameters
country = "ES"
year_start = 2009
year_end = 2024



#################### 4-panel bar chart: space, water, cooking, and total residential

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_es = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_es.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_es = df_es[(df_es["year"] >= year_start) & (df_es["year"] <= year_end)].copy()
if df_es.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total residential space", "electricity residential space", "Space"),
    ("total residential water", "electricity residential water", "Water"),
    ("total residential cooking", "electricity residential cooking", "Cooking"),
    ("total residential", "electricity residential", "Residential total"),
]

required_cols = [c for pair in pairs for c in pair[:2]]
missing_cols = [c for c in required_cols if c not in df_es.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_es.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(x - center_shift, plot_df[total_col].values, width=width, color="#4E79A7", alpha=0.8, label="Total")
    ax.bar(x + center_shift, plot_df[elec_col].values, width=width, color="#F28E2B", alpha=0.8, label="Electricity")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("TWh")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} residential demand components by year", y=1.02, fontsize=14)
plt.tight_layout()


### `services`

In [ ]:
#################### Parameters
country = "ES"
year_start = 2009
year_end = 2024



#################### 4-panel bar chart: space, water, cooking, and total services

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total services space", "electricity services space", "Space"),
    ("total services water", "electricity services water", "Water"),
    ("total services cooking", "electricity services cooking", "Cooking"),
    ("total services", "electricity services", "Services total"),
]

required_cols = [c for pair in pairs for c in pair[:2]]
missing_cols = [c for c in required_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(x - center_shift, plot_df[total_col].values, width=width, color="#4E79A7", alpha=0.8, label="Total")
    ax.bar(x + center_shift, plot_df[elec_col].values, width=width, color="#F28E2B", alpha=0.8, label="Electricity")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("TWh")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} services demand components by year", y=1.02, fontsize=14)
plt.tight_layout()


### `agriculture`

It includes:

- `total agriculture electricity` = Lighting + Ventilation + Specific electricity uses + Pumping devices (electricity)
- `total agriculture heat` = Specific heat uses + Low enthalpy heat
- `total agriculture machinery` = Motor drives + Farming machine drives (diesel + biofuels) + Pumping devices (diesel + biofuels)
- `total agriculture` = "Agriculture, forestry and fishing"

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### 4-panel bar chart: electricity, heat, machinery, and total agriculture (total only)

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

cols = [
    ("total agriculture electricity", "Electricity"),
    ("total agriculture heat", "Heat"),
    ("total agriculture machinery", "Machinery"),
    ("total agriculture", "Agriculture total"),
]

required_cols = [c for c, _ in cols]
missing_cols = [c for c in required_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.62

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (col, title) in zip(axes, cols):
    ax.bar(x, plot_df[col].values, width=width, color="#4E79A7", alpha=0.85, label="Total")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} agriculture demand components by year", y=1.02, fontsize=14)
plt.tight_layout()


### `road`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024


#################### 6-panel bar chart: road demand components

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total road", "electricity road", "Road total"),
    ("total two-wheel", None, "Two-wheel"),
    ("total passenger cars", "electricity passenger cars", "Passenger cars"),
    ("total other road passenger", "electricity other road passenger", "Other road passenger"),
    ("total light duty road freight", "electricity light duty road freight", "Light duty road freight"),
    ("total heavy duty road freight", None, "Heavy duty road freight"),
]

required_cols = []
for total_col, elec_col, title in pairs:
    required_cols.append(total_col)
    if elec_col is not None:
        required_cols.append(elec_col)

missing_cols = [col for col in required_cols if col not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

overlap = 0.75
center_shift = width * (1 - overlap) / 2
x_limits = (-0.5, len(plot_df.index) - 0.5)

ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(3, 2, figsize=(18, 14), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(
        x - center_shift if elec_col is not None else x,
        plot_df[total_col].values,
        width=width,
        color="#4E79A7",
        alpha=0.8,
        label="Total",
    )

    if elec_col is not None:
        ax.bar(
            x + center_shift,
            plot_df[elec_col].values,
            width=width,
            color="#F28E2B",
            alpha=0.8,
            label="Electricity",
        )

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_xlim(*x_limits)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} road demand components by year", y=1.02, fontsize=14)
plt.tight_layout()


### `aviation`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### Stacked bar chart: aviation demand components

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

aviation_cols = [
    "total domestic aviation passenger",
    "total international aviation passenger",
    "total domestic aviation freight",
    "total international aviation freight",
]

missing_cols = [c for c in aviation_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[aviation_cols]
    .sum()
    .sort_index()
)

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6),
    color=["#4E79A7", "#F28E2B", "#59A14F", "#E15759"],
)

ax.set_title(f"{country} aviation demand components by year")
ax.set_xlabel("Year")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Components", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()


### `navigation`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### Stacked bar chart: navigation demand components

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

navigation_cols = [
    "total domestic navigation",
    "total international navigation",
]

missing_cols = [c for c in navigation_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[navigation_cols]
    .sum()
    .sort_index()
)

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color=["#4E79A7", "#F28E2B"],
)

ax.set_title(f"{country} navigation demand components by year")
ax.set_xlabel("Year")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Components", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()


### `rail`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024



#################### 3-panel bar chart: rail passenger, rail freight, and rail total

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total rail passenger", "electricity rail passenger", "Rail passenger"),
    ("total rail freight", "electricity rail freight", "Rail freight"),
    ("total rail", "electricity rail", "Rail total"),
]

required_cols = [c for pair in pairs for c in pair[:2]]
missing_cols = [c for c in required_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(x - center_shift, plot_df[total_col].values, width=width, color="#4E79A7", alpha=0.8, label="Total")
    ax.bar(x + center_shift, plot_df[elec_col].values, width=width, color="#F28E2B", alpha=0.8, label="Electricity")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

# Hide the unused 4th subplot (only 3 rail components requested)
for ax in axes[len(pairs):]:
    ax.axis("off")

fig.suptitle(f"{country} rail demand components by year", y=1.02, fontsize=14)
plt.tight_layout()


## `co2_totals.csv`

CO2 totals are for base emissions year defined in the pypsa configuration file (typically 1990):

energy:
  base_emissions_year: 1990

Load the file and preview its content.

In [ ]:
file = "co2_totals.csv"

df_co2_totals = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df_co2_totals.head()

In [ ]:
#################### Parameters
country = "ES"
figsize = (14, 7)
title_fontsize = 15
axis_fontsize = 12
tick_fontsize = 11
legend_fontsize = 10

#################### Stacked bar chart: CO2 totals by component
country_col = "Country_code" if "Country_code" in df_co2_totals.columns else "country"
if country_col not in df_co2_totals.columns:
    raise KeyError("The DataFrame must contain either 'Country_code' or 'country'.")

df_country = df_co2_totals[df_co2_totals[country_col] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country_col} == '{country}'.")

x_label = country
if "year" in df_country.columns:
    df_country = df_country.sort_values("year")
    selected_year = df_country["year"].iloc[-1]
    df_country = df_country[df_country["year"] == selected_year].copy()
    x_label = f"{country} ({selected_year})"

component_cols = [
    col for col in df_country.columns
    if col not in {country_col, "year"}
]

raw_values = df_country[component_cols].sum(axis=0)
if raw_values.empty:
    raise ValueError("No CO2 components are available for plotting.")

# Separate positive and negative values
positive_cols = [col for col in component_cols if raw_values[col] >= 0]
negative_cols = [col for col in component_cols if raw_values[col] < 0]

# Sort each group by magnitude (descending)
positive_cols_sorted = sorted(positive_cols, key=lambda x: raw_values[x], reverse=True)
negative_cols_sorted = sorted(negative_cols, key=lambda x: raw_values[x], reverse=True)

ordered_cols = positive_cols_sorted + negative_cols_sorted
values = raw_values[ordered_cols]

# Calculate sums
positive_sum = values[positive_cols_sorted].sum() if positive_cols_sorted else 0
negative_sum = values[negative_cols_sorted].sum() if negative_cols_sorted else 0
total_mtco2 = values.sum()

plot_df = values.to_frame().T
plot_df.index = [x_label]

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=figsize,
    width=0.6,
    colormap="tab20",
)

# Reorder legend: positives (largest first), then negatives (largest first)
legend_labels = positive_cols_sorted + negative_cols_sorted
handles, labels = ax.get_legend_handles_labels()
handle_by_label = dict(zip(labels, handles))
legend_handles = [handle_by_label[label] for label in legend_labels if label in handle_by_label]

ax.set_title(
    f"{country} CO2 totals: Positive {positive_sum:.2f} | Negative {negative_sum:.2f} | Total {total_mtco2:.2f} MtCO2",
    fontsize=title_fontsize,
)
ax.set_xlabel("")
ax.set_ylabel("MtCO2", fontsize=axis_fontsize)
ax.tick_params(axis="x", rotation=0, labelsize=tick_fontsize)
ax.tick_params(axis="y", labelsize=tick_fontsize)
ax.axhline(0, color="black", linewidth=1)
ax.grid(axis="y", alpha=0.25)
ax.legend(
    legend_handles,
    legend_labels,
    title="Components",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=legend_fontsize,
)

plt.tight_layout()


## `transport_data.csv`

In [ ]:
file = "transport_data.csv"

df_transport_data = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

print(f"Shape: {df_transport_data.shape}")
df_transport_data.head()

In [ ]:
# Side-by-side plots for transport variables by year
col_year = "year"
col_cars = "number cars"
col_eff = "average fuel efficiency"

missing = [c for c in [col_year, col_cars, col_eff] if c not in df_transport_data.columns]
if missing:
    raise KeyError(f"Missing required columns in transport_data.csv: {missing}")

plot_df = df_transport_data[[col_year, col_cars, col_eff]].copy().sort_values(col_year)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(plot_df[col_year], plot_df[col_cars].div(1e6), marker="o", linewidth=1.5)
ax1.set_title("Number cars by year")
ax1.set_xlabel("year")
ax1.set_ylabel("number cars [Mill.]")
ax1.grid(True, linestyle="--", alpha=0.4)

ax2.plot(plot_df[col_year], plot_df[col_eff], marker="o", linewidth=1.5, color="tab:orange")
ax2.set_title("Average fuel efficiency [MWh/100km] by year")
ax2.set_xlabel("year")
ax2.set_ylabel("average fuel efficiency")
ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()

## `district_heat_share.csv`

In [ ]:
file = "district_heat_share.csv"

df_district_heat_share = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

print(f"Shape: {df_district_heat_share.shape}")
df_district_heat_share.head()

In [ ]:
# District heat share by year (single plot)
# Expected columns: country, 1990, 1991, ...

country_col = df_district_heat_share.columns[0]
year_cols = list(df_district_heat_share.columns[1:])

if not year_cols:
    raise ValueError("district_heat_share.csv must contain year columns after the first country column.")

# Keep only columns that can be interpreted as years
years = []
valid_year_cols = []
for c in year_cols:
    try:
        years.append(int(str(c)))
        valid_year_cols.append(c)
    except ValueError:
        continue

if not valid_year_cols:
    raise ValueError("No valid year columns found in district_heat_share.csv.")

plot_rows = df_district_heat_share[[country_col] + valid_year_cols].copy()

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

for _, row in plot_rows.iterrows():
    values = np.array(row[valid_year_cols], dtype=float)
    ax.plot(years, values, marker="o", linewidth=1.5, label=str(row[country_col]))

ax.set_title("District heat share by year")
ax.set_xlabel("year")
ax.set_ylabel("district heat share")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(title=country_col)

plt.tight_layout()

## `heating_efficiencies.csv`

In [ ]:
file = "heating_efficiencies.csv"

df_heating_efficiencies = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

print(f"Shape: {df_heating_efficiencies.shape}")
df_heating_efficiencies.head()

In [ ]:
# Heating efficiencies by country: one plot per country
country_col = "country" if "country" in df_heating_efficiencies.columns else None
year_col = "year" if "year" in df_heating_efficiencies.columns else None

if country_col is None or year_col is None:
    raise KeyError("heating_efficiencies.csv must contain 'country' and 'year' columns.")

value_cols = [
    c for c in df_heating_efficiencies.columns
    if c not in [country_col, year_col] and np.issubdtype(df_heating_efficiencies[c].dtype, np.number)
]

if not value_cols:
    raise ValueError("No numeric columns found to plot in heating_efficiencies.csv.")

countries = sorted(df_heating_efficiencies[country_col].dropna().unique())

for ctry in countries:
    df_country = (
        df_heating_efficiencies[df_heating_efficiencies[country_col] == ctry]
        .copy()
        .sort_values(year_col)
    )

    fig, ax = plt.subplots(figsize=(11, 5))

    for col in value_cols:
        ax.plot(df_country[year_col], df_country[col], marker="o", linewidth=1.5, label=col)

    ax.set_title(f"Heating efficiencies by year - {ctry}")
    ax.set_xlabel("year")
    ax.set_ylabel("efficiency")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(title="Variables", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()